# 🏦 Análise de Incidentes de IA em Serviços Financeiros

**Projeto**: Análise de Incidentes de IA em Serviços Financeiros — Viés Algorítmico, Risco Operacional e Governança  
**Metodologia**: CRISP-DM (6 fases)  
**Instituição**: PUC-SP | **Disciplina**: Projeto Integrador — 5º Semestre  
**Fonte de dados**: AI Incident Database (AIID) — https://incidentdatabase.ai

---

## 🗺️ Estrutura deste notebook unificado

| Fase CRISP-DM | Seção | Conteúdo |
|---|---|---|
| Data Understanding + Preparation | **PARTE 1** | Coleta, limpeza, feature engineering, banco SQLite |
| Evaluation | **PARTE 2** | Estatística descritiva e 4 testes de hipóteses |
| Modeling | **PARTE 3** | Regressão Logística, Random Forest, XGBoost, SHAP |
| Deployment | **PARTE 4** | API RESTful com Flask + testes automatizados |

## 📦 Arquivos gerados ao longo do notebook

- `incidents_finance_filtered.csv` — dataset processado
- `ai_finance_incidents.db` — banco SQLite com 3 tabelas relacionadas
- `models/severity_classifier.pkl` — modelo de severidade
- `models/investigation_classifier.pkl` — modelo de investigação regulatória
- `models/features_severity.pkl` / `features_investigation.pkl` — listas de features


## 📐 Estrutura do notebook único

├── PARTE 1 — Data Understanding + Preparation
│   ├── Célula de instalação de dependências
│   ├── Importações centralizadas (todos os pacotes)
│   ├── Coleta via GraphQL com fallback CSV
│   ├── Exploração + qualidade
│   ├── Filtragem temática (financeiro)
│   ├── Limpeza + padronização
│   ├── Feature engineering (8 variáveis)
│   ├── Visualização 2×2
│   ├── Salvamento CSV
│   └── Banco SQLite (3 tabelas + verificação JOIN)
│
├── PARTE 2 — Evaluation (Testes de Hipóteses)
│   ├── Carregamento + normalização defensiva
│   ├── Descritiva: categóricas + temporal + governança
│   ├── H1 — Qui-quadrado de aderência
│   ├── H2 — Qui-quadrado de independência + taxa por segmento
│   ├── H3 — Qui-quadrado + Regressão Logística + Odds Ratio
│   ├── H4 — Spearman + Regressão Linear + gráfico de tendência
│   └── Síntese para gestão de risco
│
├── PARTE 3 — Modeling (Machine Learning)
│   ├── Carregamento + padronização defensiva
│   ├── Feature selection + One-Hot Encoding
│   ├── Targets (binary + multiclass)
│   ├── Modelo 1 (Severidade): LR + RF + XGBoost
│   ├── Comparação de métricas + validação cruzada
│   ├── Matriz de confusão + curvas ROC
│   ├── Feature Importance + SHAP
│   ├── Modelo 2 (Investigação): XGBoost
│   └── Salvamento dos 4 artefatos .pkl
│
└── PARTE 4 — Deployment (API RESTful)
    ├── Helpers SQLite + carregamento de modelos
    ├── prepare_model_input() com docstring
    ├── 9 endpoints Flask definidos
    ├── Testes automatizados (test_client)
    └── Geração do app.py standalone

---
# 📂 PARTE 1 — Exploração e Preparação de Dados

**Fase CRISP-DM**: Data Understanding + Data Preparation

Esta seção implementa a coleta dos dados via API GraphQL oficial do AIID com fallback para CSV local, realiza exploração inicial, aplica filtragem temática para o setor financeiro, executa feature engineering com 8 variáveis derivadas e constrói o banco SQLite relacional com 3 tabelas.

---

## 1️⃣ Instalação de dependências

**O que faz**: Instala bibliotecas adicionais que podem não estar presentes no Colab por padrão.

**Por que é importante**: Garante que xgboost, shap e flask-cors estejam disponíveis em todos os ambientes antes de qualquer importação.

In [ ]:
!pip install -q xgboost shap flask flask-cors joblib

## 2️⃣ Importação de bibliotecas

**O que faz**: Importa todos os pacotes necessários para as quatro fases do projeto em um único bloco centralizado.

**Por que é importante**: Centralizar os imports evita erros de dependência ao longo do notebook e facilita manutenção.

In [ ]:
import os
import re
import json
import sqlite3
import warnings
import joblib
import requests

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from scipy import stats
from scipy.stats import chi2_contingency
import statsmodels.api as sm

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)

import xgboost as xgb
import shap

from flask import Flask, request, jsonify
from flask_cors import CORS
from datetime import datetime

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1200)

RANDOM_STATE = 42

print('='*80)
print('✅ BIBLIOTECAS IMPORTADAS COM SUCESSO')
print('='*80)
print(f'📌 Pandas   : {pd.__version__}')
print(f'📌 NumPy    : {np.__version__}')
print(f'📌 XGBoost  : {xgb.__version__}')
print('\n🚀 Ambiente configurado e pronto para análise!')

## 3️⃣ Carregamento dos dados — API AIID com fallback CSV

**O que faz**: Consulta o endpoint GraphQL oficial do AI Incident Database (AIID). Se a API estiver indisponível, carrega automaticamente o cache local `incidents.csv`.

**Fonte dos dados**:
- Endpoint oficial: `https://incidentdatabase.ai/api/graphql`
- Fallback: `incidents.csv` (download via Kaggle: https://www.kaggle.com/datasets/konradb/ai-incident-database)

**Caso de uso**: Garante reprodutibilidade mesmo em ambientes sem acesso à internet ou com limite de requisições.

In [ ]:
AIID_GRAPHQL_URL = "https://incidentdatabase.ai/api/graphql"
CACHE_FILE_JSON  = "aiid_incidents_raw.json"
CACHE_FILE_CSV   = "aiid_incidents_raw.csv"
LOCAL_CSV        = "incidents.csv"

GRAPHQL_QUERY = """
query {
  reports {
    report_number
    title
    text
    date_published
    url
  }
}
"""

def normalize_aiid_response(records):
    """Normaliza a resposta da API GraphQL para um DataFrame padronizado."""
    df = pd.DataFrame(records)
    rename_map = {
        "report_number": "incident_id",
        "text": "description",
        "date_published": "date"
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
    for col, default in [('incident_id', None), ('title', 'Untitled'),
                         ('description', ''), ('date', pd.NaT)]:
        if col not in df.columns:
            df[col] = default
    if df['incident_id'].isnull().all():
        df['incident_id'] = range(1, len(df) + 1)
    return df

# — Tentativa 1: API GraphQL —
try:
    resp = requests.post(AIID_GRAPHQL_URL, json={"query": GRAPHQL_QUERY}, timeout=60)
    resp.raise_for_status()
    payload = resp.json()
    if "errors" in payload:
        raise ValueError(f"Erros na API: {payload['errors']}")
    records = payload.get("data", {}).get("reports", [])
    if not records:
        raise ValueError("API retornou zero registros.")
    df_original = normalize_aiid_response(records)
    with open(CACHE_FILE_JSON, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    df_original.to_csv(CACHE_FILE_CSV, index=False)
    source = "API GraphQL oficial do AIID"

# — Tentativa 2: Cache JSON —
except Exception as e1:
    print(f"⚠️ API indisponível: {e1}")
    if os.path.exists(CACHE_FILE_JSON):
        print("🔄 Carregando cache JSON local...")
        with open(CACHE_FILE_JSON, encoding="utf-8") as f:
            records = json.load(f)
        df_original = normalize_aiid_response(records)
        source = "Cache JSON local"

    # — Tentativa 3: CSV local —
    elif os.path.exists(CACHE_FILE_CSV):
        print("🔄 Carregando cache CSV local...")
        df_original = pd.read_csv(CACHE_FILE_CSV, low_memory=False)
        source = "Cache CSV local"

    elif os.path.exists(LOCAL_CSV):
        print("🔄 Carregando arquivo CSV do Kaggle...")
        df_original = pd.read_csv(LOCAL_CSV, low_memory=False)
        source = "CSV local do Kaggle"

    else:
        raise FileNotFoundError(
            "Nenhuma fonte de dados disponível.\n"
            "Faça upload de 'incidents.csv' ou garanta acesso à internet."
        )

print('='*80)
print(f'📊 DATASET CARREGADO — {source.upper()}')
print('='*80)
print(f'✅ Total de registros  : {len(df_original):,}')
print(f'📋 Total de colunas    : {len(df_original.columns)}')
print(f'💾 Memória             : {df_original.memory_usage(deep=True).sum()/1024**2:.2f} MB')
display(df_original.head(3))

## 4️⃣ Exploração inicial e qualidade dos dados

**O que faz**: Inspeciona estrutura, tipos, valores ausentes e duplicatas do dataset bruto.

**Por que é importante**: Conhecer a qualidade dos dados antes da limpeza evita vieses e erros propagados nas etapas seguintes.

In [ ]:
print('='*80)
print('📋 ESTRUTURA DO DATASET')
print('='*80)
print(f'\n🔍 Colunas ({len(df_original.columns)} total):')
for i, col in enumerate(df_original.columns, 1):
    print(f'  {i:2d}. {col}')

df_original.info()

# Valores ausentes
missing = pd.DataFrame({
    'Coluna': df_original.columns,
    'Ausentes': df_original.isnull().sum().values,
    'Percentual_%': (df_original.isnull().sum().values / len(df_original) * 100).round(2)
}).query('Ausentes > 0').sort_values('Ausentes', ascending=False)

print('\n' + '='*80)
print('🔍 VALORES AUSENTES')
print('='*80)
if len(missing) > 0:
    print(missing.to_string(index=False))
else:
    print('✅ Nenhum valor ausente encontrado.')

# Duplicatas
id_col = next((c for c in ['incident_id', 'report_number'] if c in df_original.columns), None)
if id_col:
    dups = df_original.duplicated(subset=[id_col]).sum()
    print(f'\n🔄 Duplicatas em [{id_col}]: {dups}')

## 5️⃣ Filtragem temática — Setor financeiro

**O que faz**: Seleciona apenas incidentes relacionados a bancos, fintechs, crédito, trading e demais subdomínios financeiros usando correspondência de palavras-chave no texto.

**Caso de uso**: Delimitar o escopo do projeto ao setor de interesse, reduzindo ruído analítico.

In [ ]:
FINANCIAL_KEYWORDS = [
    'finance', 'financial', 'banking', 'bank', 'fintech',
    'credit', 'loan', 'mortgage', 'debt', 'lending',
    'fraud detection', 'anti-money laundering', 'aml',
    'trading', 'algorithmic trading', 'high-frequency', 'flash crash',
    'investment', 'robo-advisor', 'portfolio',
    'payment', 'transaction', 'insurance', 'underwriting',
    'risk assessment', 'wealth management', 'hedge fund',
    'stock', 'equity', 'derivative', 'securities'
]

def is_financial(row):
    """Retorna True se o incidente pertence ao setor financeiro."""
    text = (str(row.get('title', '')) + ' ' + str(row.get('description', ''))).lower()
    return any(kw in text for kw in FINANCIAL_KEYWORDS)

df_finance = df_original[df_original.apply(is_financial, axis=1)].copy()

print('='*80)
print('🎯 FILTRAGEM TEMÁTICA — SETOR FINANCEIRO')
print('='*80)
print(f'📊 Incidentes originais : {len(df_original):,}')
print(f'💰 Incidentes financeiros: {len(df_finance):,}')
print(f'📈 Percentual            : {len(df_finance)/len(df_original)*100:.2f}%')

if len(df_finance) < 20:
    print('⚠️ Amostra muito pequena. Verifique keywords e dataset de entrada.')
else:
    print('✅ Filtragem concluída com sucesso!')

## 6️⃣ Limpeza e padronização

**O que faz**: Renomeia colunas, converte datas, cria o campo de texto unificado, remove duplicatas e preenche valores essenciais ausentes.

**Por que é importante**: Padronização evita erros nos notebooks seguintes e garante consistência no banco SQLite.

In [ ]:
# Renomear colunas para snake_case padronizado
rename_map = {
    'date': 'occurred_date', 'date_published': 'occurred_date',
    'description': 'summary',
    'Alleged deployer of AI system': 'deployer',
    'Alleged developer of AI system': 'developer',
    'Alleged harmed or nearly harmed parties': 'harmed_parties'
}
for old, new in rename_map.items():
    if old in df_finance.columns and new not in df_finance.columns:
        df_finance = df_finance.rename(columns={old: new})

# Converter datas e extrair ano
date_col = next((c for c in ['occurred_date'] if c in df_finance.columns), None)
if date_col:
    df_finance[date_col] = pd.to_datetime(df_finance[date_col], errors='coerce')
    df_finance['year'] = df_finance[date_col].dt.year
else:
    df_finance['year'] = np.nan

# Campo de texto unificado (title + summary)
df_finance['title']   = df_finance.get('title', pd.Series(dtype=str)).fillna('')
df_finance['summary'] = df_finance.get('summary', pd.Series(dtype=str)).fillna('')
df_finance['text'] = (df_finance['title'] + ' ' + df_finance['summary']).str.lower().str.strip()

# Remover duplicatas
id_col = next((c for c in ['incident_id', 'report_number'] if c in df_finance.columns), None)
if id_col and 'incident_id' not in df_finance.columns:
    df_finance = df_finance.rename(columns={id_col: 'incident_id'})
if 'incident_id' in df_finance.columns:
    before = len(df_finance)
    df_finance = df_finance.drop_duplicates(subset=['incident_id'], keep='first')
    print(f'🔄 Duplicatas removidas: {before - len(df_finance)}')

print('='*80)
print('✅ LIMPEZA CONCLUÍDA')
print('='*80)
print(f'📊 Registros finais : {len(df_finance):,}')
if df_finance['year'].notna().any():
    print(f'📅 Período          : {int(df_finance["year"].min())} – {int(df_finance["year"].max())}')
display(df_finance[['incident_id','occurred_date','year','title']].head(3))

## 7️⃣ Engenharia de features — 8 variáveis derivadas

**O que faz**: Cria variáveis analíticas estruturadas a partir do texto livre dos incidentes, usando classificação por palavras-chave.

**Variáveis criadas**:
1. `application_type` — tipo de aplicação de IA (credit_scoring, fraud_detection, algorithmic_trading…)
2. `incident_type` — natureza do problema (algorithmic_bias, operational_failure…)
3. `customer_segment` — segmento de cliente afetado
4. `severity_level` — severidade (low → critical)
5–8. Flags binários de governança: `regulatory_investigation`, `fine_imposed`, `policy_change`, `third_party_audit`

**Caso de uso**: Transformar texto não estruturado em variáveis categóricas prontas para estatística e ML.

In [ ]:
def classify_application_type(text):
    t = str(text).lower()
    if any(k in t for k in ['credit scor','loan','lending','mortgage','credit decision','credit approv']):
        return 'credit_scoring'
    elif any(k in t for k in ['fraud','aml','anti-money laundering','suspicious transaction']):
        return 'fraud_detection'
    elif any(k in t for k in ['trading','high-frequency','flash crash','algorithmic trad']):
        return 'algorithmic_trading'
    elif any(k in t for k in ['robo-advis','robo advis','investment advisor','portfolio manag']):
        return 'robo_advisor'
    elif any(k in t for k in ['risk assessment','underwriting','insurance','actuarial']):
        return 'risk_assessment'
    elif any(k in t for k in ['chatbot','customer service','virtual assistant']):
        return 'customer_service'
    return 'other_finance'

def classify_incident_type(text):
    t = str(text).lower()
    if any(k in t for k in ['bias','biased','discriminat','unfair','gender','racial','disproportion']):
        return 'algorithmic_bias'
    elif any(k in t for k in ['flash crash','market disruption','volatility','market crash']):
        return 'market_disruption'
    elif any(k in t for k in ['crash','failure','outage','malfunction','error','bug','glitch']):
        return 'operational_failure'
    elif any(k in t for k in ['breach','leak','hack','data exposure','privacy violation']):
        return 'data_breach'
    elif any(k in t for k in ['regulat','violation','fine','sanction','compliance','illegal']):
        return 'regulatory_violation'
    return 'other'

def classify_customer_segment(text):
    t = str(text).lower()
    if any(k in t for k in ['small business','sme','small and medium','startup']):
        return 'sme'
    elif any(k in t for k in ['retail','consumer','individual','customer','personal']):
        return 'retail'
    elif any(k in t for k in ['corporate','enterprise','large business','institution']):
        return 'corporate'
    elif any(k in t for k in ['underserved','minority','low-income','vulnerable','disadvantaged']):
        return 'underserved'
    return 'general'

def classify_severity(text):
    t = str(text).lower()
    if any(k in t for k in ['bankrupt','systemic','shutdown','suspended','billion','catastrophic']):
        return 'critical'
    elif any(k in t for k in ['investigation','lawsuit','fine','penalty','significant loss','major']):
        return 'high'
    elif any(k in t for k in ['complaint','concern','review','moderate','issue']):
        return 'medium'
    return 'low'

def extract_governance_flags(text):
    t = str(text).lower()
    return {
        'regulatory_investigation': int(any(k in t for k in ['investigation','inquiry','probe','sec','regulat'])),
        'fine_imposed':             int(any(k in t for k in ['fine','penalty','sanction','settlement'])),
        'policy_change':            int(any(k in t for k in ['policy change','suspended','discontinued','halted'])),
        'third_party_audit':        int(any(k in t for k in ['audit','independent review','third-party']))
    }

# Aplicar features
df_finance['application_type']  = df_finance['text'].apply(classify_application_type)
df_finance['incident_type']     = df_finance['text'].apply(classify_incident_type)
df_finance['customer_segment']  = df_finance['text'].apply(classify_customer_segment)
df_finance['severity_level']    = df_finance['text'].apply(classify_severity)

gov_df = df_finance['text'].apply(extract_governance_flags).apply(pd.Series)
df_finance = pd.concat([df_finance, gov_df], axis=1)

print('✅ 8 variáveis derivadas criadas:')
for v in ['application_type','incident_type','customer_segment','severity_level',
          'regulatory_investigation','fine_imposed','policy_change','third_party_audit']:
    n = df_finance[v].sum() if df_finance[v].dtype in ['int64','float64'] else df_finance[v].nunique()
    label = f'({n} categorias)' if df_finance[v].dtype == object else f'({int(n)} com flag=1)'
    print(f'  ✓ {v} {label}')

## 8️⃣ Visualização das variáveis derivadas

**O que faz**: Gera painel 2×2 com a distribuição de cada variável categórica derivada.

**Caso de uso**: Validação visual de que as classificações fazem sentido antes de avançar para análise.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('📊 Distribuição das Variáveis Derivadas — Incidentes Financeiros de IA',
             fontsize=15, fontweight='bold', y=0.998)

for ax, col, color, title in [
    (axes[0,0], 'application_type',  'steelblue',      'Tipo de Aplicação de IA'),
    (axes[0,1], 'incident_type',     'coral',           'Tipo de Incidente'),
    (axes[1,0], 'customer_segment',  'mediumseagreen',  'Segmento de Cliente'),
    (axes[1,1], 'severity_level',    None,              'Nível de Severidade'),
]:
    counts = df_finance[col].value_counts()
    if col == 'severity_level':
        order  = ['low','medium','high','critical']
        counts = counts.reindex([s for s in order if s in counts.index], fill_value=0)
        colors = ['#90EE90','#FFD700','#FF8C00','#DC143C'][:len(counts)]
        bars = ax.bar(range(len(counts)), counts.values, color=colors)
        ax.set_xticks(range(len(counts)))
        ax.set_xticklabels(counts.index)
        for b in bars:
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.2,
                    str(int(b.get_height())), ha='center', fontsize=9)
    else:
        ax.barh(counts.index, counts.values, color=color)
        for i, v in enumerate(counts.values):
            ax.text(v+0.1, i, str(v), va='center', fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(axis='x' if col != 'severity_level' else 'y', alpha=0.3)

plt.tight_layout()
plt.show()
print('✅ Visualizações geradas.')

## 9️⃣ Salvamento do CSV processado

**O que faz**: Persiste o dataset limpo e enriquecido como `incidents_finance_filtered.csv` para uso nas próximas seções.

In [ ]:
COLS_TO_SAVE = [
    'incident_id','occurred_date','title','summary','year','text',
    'application_type','incident_type','customer_segment','severity_level',
    'regulatory_investigation','fine_imposed','policy_change','third_party_audit'
]
available = [c for c in COLS_TO_SAVE if c in df_finance.columns]
df_final = df_finance[available].copy()
df_final.to_csv('incidents_finance_filtered.csv', index=False)

print('='*80)
print('💾 CSV SALVO: incidents_finance_filtered.csv')
print('='*80)
print(f'📊 Registros : {len(df_final):,}')
print(f'📋 Colunas   : {len(df_final.columns)}')
print(f'\nColunas: {", ".join(available)}')

## 🔟 Criação do banco de dados SQLite (3 tabelas)

**O que faz**: Cria e popula o banco `ai_finance_incidents.db` com 3 tabelas relacionadas via chaves estrangeiras.

**Caso de uso**: Requisito do briefing (≥3 tabelas) e base da API RESTful.

| Tabela | Descrição |
|---|---|
| `incidents` | Tabela principal: todos os incidentes e suas features |
| `financial_impacts` | Impacto financeiro estimado por incidente |
| `regulatory_responses` | Respostas de governança e investigações por incidente |

In [ ]:
DB_FILE = 'ai_finance_incidents.db'
conn   = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

# ── DROP + CREATE ──────────────────────────────────────────────────────────────
for tbl in ('regulatory_responses','financial_impacts','incidents'):
    cursor.execute(f'DROP TABLE IF EXISTS {tbl}')

cursor.execute('''
CREATE TABLE incidents (
    incident_id       INTEGER PRIMARY KEY,
    title             TEXT    NOT NULL,
    summary           TEXT,
    occurred_date     DATE,
    year              INTEGER,
    application_type  TEXT,
    customer_segment  TEXT,
    incident_type     TEXT,
    severity_level    TEXT,
    text              TEXT
)''')

cursor.execute('''
CREATE TABLE financial_impacts (
    impact_id          INTEGER PRIMARY KEY AUTOINCREMENT,
    incident_id        INTEGER NOT NULL,
    severity_level     TEXT,
    estimated_loss     TEXT,
    impact_description TEXT,
    FOREIGN KEY (incident_id) REFERENCES incidents(incident_id)
)''')

cursor.execute('''
CREATE TABLE regulatory_responses (
    response_id              INTEGER PRIMARY KEY AUTOINCREMENT,
    incident_id              INTEGER NOT NULL,
    regulatory_investigation INTEGER DEFAULT 0,
    fine_imposed             INTEGER DEFAULT 0,
    policy_change            INTEGER DEFAULT 0,
    third_party_audit        INTEGER DEFAULT 0,
    response_description     TEXT,
    FOREIGN KEY (incident_id) REFERENCES incidents(incident_id)
)''')

# ── POPULAR incidents ──────────────────────────────────────────────────────────
inc_cols = ['incident_id','title','summary','occurred_date','year',
            'application_type','customer_segment','incident_type','severity_level','text']
inc_data = df_final[[c for c in inc_cols if c in df_final.columns]].copy()
if 'occurred_date' in inc_data.columns:
    inc_data['occurred_date'] = inc_data['occurred_date'].astype(str)
inc_data.to_sql('incidents', conn, if_exists='append', index=False)

# ── POPULAR financial_impacts ──────────────────────────────────────────────────
fi = df_final[['incident_id','severity_level']].copy()
fi['estimated_loss']     = 'Not reported'
fi['impact_description'] = df_final['summary'].fillna('').str[:200]
fi.to_sql('financial_impacts', conn, if_exists='append', index=False)

# ── POPULAR regulatory_responses ──────────────────────────────────────────────
rr_cols = [c for c in ['incident_id','regulatory_investigation',
                        'fine_imposed','policy_change','third_party_audit']
           if c in df_final.columns]
rr = df_final[rr_cols].copy()
rr['response_description'] = 'Extraído do texto do incidente'
rr.to_sql('regulatory_responses', conn, if_exists='append', index=False)

conn.commit()

# ── VERIFICAÇÃO ────────────────────────────────────────────────────────────────
print('='*80)
print(f'🗄️  BANCO CRIADO: {DB_FILE}')
print('='*80)
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
for (tbl,) in cursor.fetchall():
    cursor.execute(f'SELECT COUNT(*) FROM {tbl}')
    n = cursor.fetchone()[0]
    print(f'  ✓ {tbl:30s}: {n:,} registros')

# Consulta JOIN de exemplo
print('\n🔗 Exemplo JOIN — Incidentes críticos/altos com investigação:')
q = '''
SELECT i.incident_id, i.severity_level, r.regulatory_investigation
FROM incidents i
JOIN regulatory_responses r USING(incident_id)
WHERE i.severity_level IN ("high","critical") LIMIT 5
'''
display(pd.read_sql_query(q, conn))
conn.close()
print('\n✅ PARTE 1 CONCLUÍDA — CSV e banco SQLite prontos!')

---
# 📊 PARTE 2 — Análise Estatística e Testes de Hipóteses

**Fase CRISP-DM**: Evaluation

Esta seção carrega os artefatos da Parte 1 e realiza análise descritiva completa seguida de 4 testes de hipóteses estatísticas, respondendo às questões de pesquisa do projeto.

---

## 1️⃣ Carregamento dos dados processados

**O que faz**: Carrega `incidents_finance_filtered.csv` e `ai_finance_incidents.db` gerados na Parte 1.

**Justificativa metodológica**: A análise estatística opera sobre a camada já tratada, garantindo rastreabilidade.

In [ ]:
CSV_FILE = 'incidents_finance_filtered.csv'
DB_FILE  = 'ai_finance_incidents.db'

df = pd.read_csv(CSV_FILE)
conn = sqlite3.connect(DB_FILE)

# Normalização de nomes — compatibilidade entre versões
rename_candidates = {
    'incidentid':'incident_id','occurreddate':'occurred_date',
    'applicationtype':'application_type','incidenttype':'incident_type',
    'customersegment':'customer_segment','severitylevel':'severity_level',
    'regulatoryinvestigation':'regulatory_investigation',
    'fineimposed':'fine_imposed','policychange':'policy_change',
    'thirdpartyaudit':'third_party_audit'
}
for old, new in rename_candidates.items():
    if old in df.columns and new not in df.columns:
        df = df.rename(columns={old: new})

if 'occurred_date' in df.columns:
    df['occurred_date'] = pd.to_datetime(df['occurred_date'], errors='coerce')

print('='*80)
print('📊 DADOS CARREGADOS')
print('='*80)
print(f'📌 Total de incidentes : {len(df):,}')
if 'year' in df.columns and df['year'].notna().any():
    print(f'📅 Período             : {int(df["year"].min())} – {int(df["year"].max())}')
display(df[['incident_id','application_type','incident_type','customer_segment','severity_level']].head(5))

## 2️⃣ Análise descritiva

**O que faz**: Examina distribuições univariadas, comportamento temporal e respostas de governança.

**Caso de uso**: Visão inicial quantitativa antes dos testes inferenciais.

In [ ]:
# 2.1 Variáveis categóricas
cat_vars = [c for c in ['application_type','incident_type','customer_segment','severity_level']
            if c in df.columns]

print('='*80)
print('📊 ANÁLISE DESCRITIVA — VARIÁVEIS CATEGÓRICAS')
print('='*80)
for var in cat_vars:
    counts = df[var].fillna('missing').value_counts()
    pcts   = (counts / len(df) * 100).round(2)
    tbl = pd.DataFrame({'Categoria': counts.index,
                        'Frequência': counts.values, '%': pcts.values})
    print(f'\n{"─"*60}\n{var.upper()}\n{"─"*60}')
    print(tbl.to_string(index=False))
    print(f'  → {df[var].nunique()} categorias distintas')

In [ ]:
# 2.2 Série temporal
if 'year' in df.columns:
    ts = df.groupby('year').size().reset_index(name='n').sort_values('year')
    print('='*80)
    print('📅 ANÁLISE TEMPORAL')
    print('='*80)
    print(ts.to_string(index=False))
    print(f'\n  Média/ano  : {ts["n"].mean():.1f}')
    print(f'  Mediana/ano: {ts["n"].median():.1f}')
    print(f'  Pico       : {int(ts.loc[ts["n"].idxmax(),"year"])} ({ts["n"].max()} incidentes)')
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(ts['year'], ts['n'], marker='o', linewidth=2.5, color='steelblue')
    ax.fill_between(ts['year'], ts['n'], alpha=0.15, color='steelblue')
    ax.set_title('Incidentes de IA em Finanças por Ano', fontsize=14, fontweight='bold')
    ax.set_xlabel('Ano'); ax.set_ylabel('Número de incidentes')
    ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# 2.3 Governança
gov_cols = [c for c in ['regulatory_investigation','fine_imposed','policy_change','third_party_audit']
            if c in df.columns]
if gov_cols:
    gov_tbl = pd.DataFrame({
        'Flag'       : gov_cols,
        'Total'      : [int(df[c].fillna(0).sum()) for c in gov_cols],
        'Percentual_%': [round(df[c].fillna(0).mean()*100, 2) for c in gov_cols]
    })
    print('='*80)
    print('🏛️  RESPOSTAS DE GOVERNANÇA')
    print('='*80)
    print(gov_tbl.to_string(index=False))
    fig, ax = plt.subplots(figsize=(9,4))
    ax.bar(gov_tbl['Flag'], gov_tbl['Total'], color=['#4C72B0','#DD8452','#55A868','#C44E52'])
    ax.set_title('Frequência de respostas de governança', fontsize=13, fontweight='bold')
    ax.set_xlabel('Flag'); ax.set_ylabel('Incidentes')
    plt.xticks(rotation=15); ax.grid(axis='y', alpha=0.3)
    for bar in ax.patches:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                str(int(bar.get_height())), ha='center', fontsize=10)
    plt.tight_layout(); plt.show()

## 3️⃣ Testes de hipóteses

**O que faz**: Executa 4 testes estatísticos para avaliar as questões de pesquisa do projeto.

**Nível de significância**: α = 0,05

In [ ]:
alpha = 0.05
print(f'Nível de significância adotado: α = {alpha}')

### H1 — Concentração por tipo de aplicação

**Pergunta**: Os incidentes de IA estão concentrados em certos tipos de aplicação financeira?

- **H₀**: Distribuição uniforme entre os tipos de aplicação.
- **H₁**: Há concentração significativa em alguns tipos.
- **Teste**: Qui-quadrado de aderência (Goodness of Fit).

In [ ]:
if 'application_type' in df.columns:
    app_counts = df['application_type'].value_counts()
    observed   = app_counts.values
    expected   = np.repeat(observed.mean(), len(observed))

    chi2_h1, p_h1 = stats.chisquare(f_obs=observed, f_exp=expected)

    print('='*80)
    print('🧪 H1 — CONCENTRAÇÃO POR TIPO DE APLICAÇÃO')
    print('='*80)
    print(app_counts.to_frame('Frequência'))
    print(f'\nQui-quadrado : {chi2_h1:.4f}')
    print(f'Valor-p      : {p_h1:.6f}')
    print(f'Decisão      : {"✅ REJEITA H₀" if p_h1 < alpha else "ℹ️  NÃO rejeita H₀"}')
    if p_h1 < alpha:
        top = app_counts.index[0]
        print(f'\n💡 A aplicação mais frequente é "{top}" ({app_counts.iloc[0]} incidentes).')
        print('   Gestores de risco devem intensificar controles nessa categoria.')
    # Gráfico
    fig, ax = plt.subplots(figsize=(10,5))
    ax.bar(app_counts.index, app_counts.values, color='steelblue')
    ax.axhline(observed.mean(), color='red', linestyle='--', label='Média esperada (H₀)')
    ax.set_title(f'H1: Distribuição por Tipo de Aplicação (p={p_h1:.4f})',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Tipo'); ax.set_ylabel('Frequência')
    plt.xticks(rotation=30, ha='right'); ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('⚠️ Coluna application_type não encontrada.')

### H2 — Viés por segmento de cliente

**Pergunta**: Incidentes de viés algorítmico afetam desproporcionalmente algum segmento?

- **H₀**: A ocorrência de viés é independente do segmento de cliente.
- **H₁**: Existe associação entre viés algorítmico e segmento de cliente.
- **Teste**: Qui-quadrado de independência.

In [ ]:
if 'incident_type' in df.columns and 'customer_segment' in df.columns:
    df_h2 = df.copy()
    df_h2['is_bias'] = (df_h2['incident_type'] == 'algorithmic_bias').astype(int)

    ct_h2 = pd.crosstab(df_h2['is_bias'], df_h2['customer_segment'])
    chi2_h2, p_h2, dof_h2, _ = chi2_contingency(ct_h2)

    print('='*80)
    print('🧪 H2 — VIÉS POR SEGMENTO DE CLIENTE')
    print('='*80)
    print('Tabela de contingência (0=sem viés | 1=com viés):')
    print(ct_h2)
    print(f'\nQui-quadrado : {chi2_h2:.4f}  |  gl={dof_h2}')
    print(f'Valor-p      : {p_h2:.6f}')
    print(f'Decisão      : {"✅ REJEITA H₀" if p_h2 < alpha else "ℹ️  NÃO rejeita H₀"}')

    # Taxa de viés por segmento
    if 1 in ct_h2.index:
        bias_rates = (ct_h2.loc[1] / ct_h2.sum(axis=0) * 100).round(2).sort_values(ascending=False)
        print('\n📊 Taxa de viés por segmento (%):')
        print(bias_rates.to_string())
        fig, ax = plt.subplots(figsize=(9,4))
        ax.bar(bias_rates.index, bias_rates.values,
               color=['#e74c3c' if v > bias_rates.mean() else '#3498db' for v in bias_rates.values])
        ax.axhline(bias_rates.mean(), color='gray', linestyle='--', label='Média')
        ax.set_title(f'H2: Taxa de Viés por Segmento (p={p_h2:.4f})', fontsize=13, fontweight='bold')
        ax.set_xlabel('Segmento'); ax.set_ylabel('%')
        ax.legend(); ax.grid(axis='y', alpha=0.3)
        plt.tight_layout(); plt.show()
else:
    print('⚠️ Colunas necessárias não encontradas.')

### H3 — Severidade e resposta regulatória

**Pergunta**: Incidentes de maior severidade resultam em investigações regulatórias mais frequentes?

- **H₀**: A investigação regulatória é independente da severidade.
- **H₁**: Alta severidade está associada a maior probabilidade de investigação.
- **Testes**: Qui-quadrado de independência + Regressão Logística.

In [ ]:
if 'severity_level' in df.columns and 'regulatory_investigation' in df.columns:
    ct_h3 = pd.crosstab(df['severity_level'], df['regulatory_investigation'])
    ct_h3.columns = ['Sem Investigação','Com Investigação']
    chi2_h3, p_h3, dof_h3, _ = chi2_contingency(ct_h3)

    print('='*80)
    print('🧪 H3 — SEVERIDADE E RESPOSTA REGULATÓRIA')
    print('='*80)
    print(ct_h3)
    print(f'\nQui-quadrado : {chi2_h3:.4f}  |  gl={dof_h3}')
    print(f'Valor-p      : {p_h3:.6f}')
    print(f'Decisão      : {"✅ REJEITA H₀" if p_h3 < alpha else "ℹ️  NÃO rejeita H₀"}')

    # Taxa de investigação por severidade
    order = ['low','medium','high','critical']
    taxa = (df.groupby('severity_level')['regulatory_investigation']
              .mean() * 100).reindex([s for s in order if s in df['severity_level'].unique()]).round(2)
    print('\n📊 Taxa de investigação por severidade (%):')
    print(taxa.to_string())

    # Regressão logística
    df_lr = df[['severity_level','regulatory_investigation']].dropna().copy()
    df_lr['high_sev'] = df_lr['severity_level'].isin(['high','critical']).astype(int)
    X_lr = sm.add_constant(df_lr['high_sev'])
    lr_res = sm.Logit(df_lr['regulatory_investigation'], X_lr).fit(disp=0)
    or_val = np.exp(lr_res.params.get('high_sev', 0))
    print(f'\n  Odds Ratio (alta vs baixa severidade): {or_val:.2f}x')

    # Gráfico
    colors_s = ['#90EE90','#FFD700','#FF8C00','#DC143C']
    fig, ax = plt.subplots(figsize=(8,5))
    ax.bar(taxa.index, taxa.values,
           color=colors_s[:len(taxa)] if len(taxa) <= 4 else colors_s)
    ax.set_title(f'H3: Taxa de Investigação por Severidade (p={p_h3:.4f})',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Severidade'); ax.set_ylabel('% com investigação')
    ax.grid(axis='y', alpha=0.3)
    for i, v in enumerate(taxa.values):
        ax.text(i, v+0.5, f'{v:.1f}%', ha='center', fontsize=10)
    plt.tight_layout(); plt.show()
else:
    print('⚠️ Colunas necessárias não encontradas.')

### H4 — Tendência temporal

**Pergunta**: Existe tendência crescente ou decrescente no número de incidentes ao longo dos anos?

- **H₀**: Não há tendência temporal significativa.
- **H₁**: Existe tendência monotônica no período analisado.
- **Testes**: Correlação de Spearman + Regressão Linear.

In [ ]:
if 'year' in df.columns:
    ts = df.groupby('year').size().reset_index(name='n').sort_values('year')
    X_t = ts['year'].values
    y_t = ts['n'].values

    rho, p_h4 = stats.spearmanr(X_t, y_t)

    X_ols = sm.add_constant(X_t)
    ols   = sm.OLS(y_t, X_ols).fit()
    slope = ols.params[1]
    r2    = ols.rsquared

    print('='*80)
    print('🧪 H4 — TENDÊNCIA TEMPORAL')
    print('='*80)
    print(f'Correlação de Spearman (ρ) : {rho:.4f}')
    print(f'Valor-p                    : {p_h4:.6f}')
    print(f'Decisão      : {"✅ REJEITA H₀" if p_h4 < alpha else "ℹ️  NÃO rejeita H₀"}')
    print(f'\nRegressão linear — slope={slope:.3f} incidentes/ano | R²={r2:.3f}')
    trend_label = '📈 CRESCENTE' if slope > 0 else '📉 DECRESCENTE'
    print(f'Direção: {trend_label}')

    # Gráfico
    trend_line = ols.params[0] + slope * X_t
    fig, ax = plt.subplots(figsize=(11,5))
    ax.plot(X_t, y_t, marker='o', linewidth=2.5, color='steelblue', label='Observado')
    ax.plot(X_t, trend_line, linestyle='--', color='red', linewidth=2,
            label=f'Tendência (R²={r2:.3f})')
    for x, y in zip(X_t, y_t):
        ax.annotate(str(int(y)), (x, y), textcoords='offset points', xytext=(0,8),
                    ha='center', fontsize=8)
    ax.set_title(f'H4: Tendência Temporal (ρ={rho:.3f}, p={p_h4:.4f})',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Ano'); ax.set_ylabel('Número de incidentes')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('⚠️ Coluna year não encontrada.')

## 4️⃣ Síntese estatística para gestão de risco

**O que faz**: Consolida os resultados dos 4 testes em linguagem de negócio para tomada de decisão.

In [ ]:
print('='*80)
print('📋 SÍNTESE DOS TESTES DE HIPÓTESES')
print('='*80)

resultados = {
    'H1 — Concentração por aplicação':  ('p_h1' in dir(), locals().get('p_h1', 1)),
    'H2 — Viés por segmento':           ('p_h2' in dir(), locals().get('p_h2', 1)),
    'H3 — Severidade e investigação':   ('p_h3' in dir(), locals().get('p_h3', 1)),
    'H4 — Tendência temporal':          ('p_h4' in dir(), locals().get('p_h4', 1)),
}
for hipotese, (exists, p) in resultados.items():
    status = '✅ CONFIRMADA' if exists and p < 0.05 else '❌ NÃO CONFIRMADA'
    pval   = f'p={p:.4f}' if exists else 'n/a'
    print(f'  {status}  {hipotese} ({pval})')

print('''
💡 RECOMENDAÇÕES:
  1. Intensificar controles na(s) aplicação(ões) com maior concentração de incidentes.
  2. Implementar auditorias de fairness nos segmentos mais afetados por viés.
  3. O sistema regulatório responde proporcionalmente à severidade — manter evidências.
  4. Monitorar tendência temporal para ajustar investimento em governança de IA.
''')
conn.close()
print('✅ PARTE 2 CONCLUÍDA!')

---
# 🤖 PARTE 3 — Modelagem de Machine Learning

**Fase CRISP-DM**: Modeling

Esta seção constrói e avalia dois modelos preditivos com XGBoost, além de Logistic Regression e Random Forest como baseline, utilizando SHAP para interpretabilidade.

**Modelos produzidos**:
- `severity_classifier` — prediz alta vs. baixa severidade
- `investigation_classifier` — prediz probabilidade de investigação regulatória

---

## 1️⃣ Carregamento e padronização defensiva

**O que faz**: Recarrega o CSV processado e garante que todos os nomes de coluna estejam no padrão `snake_case`.

In [ ]:
DATASET_FILE = 'incidents_finance_filtered.csv'
SQLITE_FILE  = 'ai_finance_incidents.db'

if not os.path.exists(DATASET_FILE):
    raise FileNotFoundError(
        f"Arquivo '{DATASET_FILE}' não encontrado. Execute a Parte 1 primeiro."
    )

df = pd.read_csv(DATASET_FILE)

# Padronização defensiva de nomes
rename_map = {
    'applicationtype':'application_type','incidenttype':'incident_type',
    'customersegment':'customer_segment','severitylevel':'severity_level',
    'regulatoryinvestigation':'regulatory_investigation','fineimposed':'fine_imposed',
    'policychange':'policy_change','thirdpartyaudit':'third_party_audit'
}
df = df.rename(columns=rename_map)

print('='*100)
print('📊 DADOS PARA MODELAGEM')
print('='*100)
print(f'✅ Arquivo  : {DATASET_FILE}')
print(f'📋 Registros: {len(df)} | Colunas: {len(df.columns)}')

expected = ['application_type','incident_type','customer_segment','year',
            'severity_level','regulatory_investigation','fine_imposed',
            'policy_change','third_party_audit']
print('\n📌 Verificação de colunas:')
for col in expected:
    status = 'OK' if col in df.columns else '⚠️ AUSENTE'
    print(f'  {status:10s} {col}')

## 2️⃣ Preparação de features e targets

**O que faz**: Seleciona as variáveis preditoras, aplica One-Hot Encoding e cria dois targets.

In [ ]:
feature_columns = ['application_type','incident_type','customer_segment',
                   'year','fine_imposed','policy_change','third_party_audit']

df_ml = df[feature_columns + ['severity_level','regulatory_investigation']].dropna().copy()

# One-Hot Encoding
categorical_features = ['application_type','incident_type','customer_segment']
df_encoded = pd.get_dummies(df_ml, columns=categorical_features, drop_first=True)

# Targets
severity_order = ['low','medium','high','critical']
df_encoded['severity_binary'] = df_encoded['severity_level'].apply(
    lambda x: 1 if x in ['high','critical'] else 0)
df_encoded['severity_multiclass'] = df_encoded['severity_level'].apply(
    lambda x: severity_order.index(x) if x in severity_order else -1)

print('='*100)
print('✅ FEATURES E TARGETS PREPARADOS')
print('='*100)
print(f'📋 Registros válidos: {len(df_encoded)}')
print(f'📊 Features após encoding: {len(df_encoded.columns)}')
print('\nseverity_binary:')
print(df_encoded['severity_binary'].value_counts().to_string())
print('\nregulatory_investigation:')
print(df_encoded['regulatory_investigation'].value_counts().to_string())

## 3️⃣ Modelo 1 — Classificação de Severidade

**Objetivo**: Prever alta vs. baixa severidade (severity_binary).

**Aplicação prática**: Triagem inicial de incidentes, priorização de resposta, encaminhamento a equipes especializadas.

### 3.1 Split treino/teste e treinamento dos 3 algoritmos

In [ ]:
X = df_encoded.drop(
    ['severity_level','severity_binary','severity_multiclass','regulatory_investigation'], axis=1)
y = df_encoded['severity_binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f'Treino: {len(X_train)} | Teste: {len(X_test)}')
print(f'Razão de classes (0/1): {(y==0).sum()/(y==1).sum():.2f}')

# ── Logistic Regression ──────────────────────────────────────────────────────
lr_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced')
lr_model.fit(X_train, y_train)
y_pred_lr      = lr_model.predict(X_test)
y_pred_proba_lr = lr_model.predict_proba(X_test)[:,1]

# ── Random Forest ────────────────────────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_split=5,
    min_samples_leaf=2, random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf      = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:,1]

# ── XGBoost ──────────────────────────────────────────────────────────────────
spw = (y_train==0).sum() / max((y_train==1).sum(), 1)
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
    random_state=RANDOM_STATE, eval_metric='logloss')
xgb_model.fit(X_train, y_train, verbose=False)
y_pred_xgb      = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:,1]

print('\n✅ Três modelos treinados.')

### 3.2 Comparação de métricas

In [ ]:
def metrics_row(name, y_true, y_pred, y_proba):
    return {
        'Modelo'   : name,
        'Accuracy' : round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall'   : round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1-Score' : round(f1_score(y_true, y_pred, zero_division=0), 4),
        'ROC-AUC'  : round(roc_auc_score(y_true, y_proba), 4)
    }

models_comparison = pd.DataFrame([
    metrics_row('Logistic Regression', y_test, y_pred_lr,  y_pred_proba_lr),
    metrics_row('Random Forest',       y_test, y_pred_rf,  y_pred_proba_rf),
    metrics_row('XGBoost',             y_test, y_pred_xgb, y_pred_proba_xgb),
])

print('='*100)
print('📊 COMPARAÇÃO — MODELO 1: SEVERIDADE')
print('='*100)
display(models_comparison.sort_values('F1-Score', ascending=False))

best = models_comparison.loc[models_comparison['F1-Score'].idxmax(), 'Modelo']
print(f'\n🏆 Melhor modelo (F1): {best}')

### 3.3 Validação cruzada, Matriz de Confusão e Curva ROC

In [ ]:
# Validação cruzada
cv_scores = cross_val_score(xgb_model, X, y, cv=5, scoring='f1')
print('='*100)
print('🔁 VALIDAÇÃO CRUZADA — XGBOOST (Severidade)')
print('='*100)
print(f'F1 por fold: {cv_scores.round(4)}')
print(f'F1 médio   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

# Visualizações
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Baixa','Alta'], yticklabels=['Baixa','Alta'])
axes[0].set_title('Matriz de Confusão — XGBoost (Severidade)', fontweight='bold')
axes[0].set_xlabel('Predito'); axes[0].set_ylabel('Real')

# Curvas ROC
for fpr, tpr, label, proba in [
    (None, None, 'Logistic Regression', y_pred_proba_lr),
    (None, None, 'Random Forest',       y_pred_proba_rf),
    (None, None, 'XGBoost',             y_pred_proba_xgb),
]:
    fpr_v, tpr_v, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[1].plot(fpr_v, tpr_v, label=f'{label} (AUC={auc:.3f})')

axes[1].plot([0,1],[0,1],'k--', label='Aleatório')
axes[1].set_title('Curvas ROC — Comparação', fontweight='bold')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend(loc='lower right')
plt.tight_layout(); plt.show()

### 3.4 Feature Importance e SHAP

In [ ]:
# Feature Importance
fi = pd.DataFrame({'Feature': X.columns,
                   'Importance': xgb_model.feature_importances_}
                 ).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
top15 = fi.head(15)
ax.barh(top15['Feature'], top15['Importance'], color='steelblue')
ax.invert_yaxis()
ax.set_title('Top 15 Features — XGBoost (Severidade)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importância')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

# SHAP
try:
    explainer   = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_test)
    shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f'⚠️ SHAP não disponível: {e}')

## 4️⃣ Modelo 2 — Predição de investigação regulatória

**Objetivo**: Prever se um incidente resultará em investigação regulatória.

**Aplicação prática**: Triagem de compliance, priorização jurídica, antecipação de exigências.

In [ ]:
X2 = df_encoded.drop(
    ['severity_level','severity_binary','severity_multiclass'], axis=1).copy()
X2['high_severity'] = df_encoded['severity_binary']
if 'regulatory_investigation' in X2.columns:
    y2 = X2.pop('regulatory_investigation')
else:
    y2 = df_encoded['regulatory_investigation']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=RANDOM_STATE, stratify=y2)

spw2 = (y2_train==0).sum() / max((y2_train==1).sum(), 1)

xgb_model_2 = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=spw2, random_state=RANDOM_STATE, eval_metric='logloss')
xgb_model_2.fit(X2_train, y2_train, verbose=False)

y2_pred       = xgb_model_2.predict(X2_test)
y2_pred_proba = xgb_model_2.predict_proba(X2_test)[:,1]

print('='*100)
print('⚖️  MODELO 2 — INVESTIGAÇÃO REGULATÓRIA')
print('='*100)
print(f'Accuracy : {accuracy_score(y2_test, y2_pred):.4f}')
print(f'Precision: {precision_score(y2_test, y2_pred, zero_division=0):.4f}')
print(f'Recall   : {recall_score(y2_test, y2_pred, zero_division=0):.4f}')
print(f'F1-Score : {f1_score(y2_test, y2_pred, zero_division=0):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y2_test, y2_pred_proba):.4f}')
print('\n', classification_report(y2_test, y2_pred,
                                    target_names=['Sem Investigação','Com Investigação']))

# Visualizações M2
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cm2 = confusion_matrix(y2_test, y2_pred)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Purples', ax=axes[0],
            xticklabels=['Sem Inv.','Com Inv.'], yticklabels=['Sem Inv.','Com Inv.'])
axes[0].set_title('Matriz de Confusão — Investigação', fontweight='bold')
axes[0].set_xlabel('Predito'); axes[0].set_ylabel('Real')
fpr2, tpr2, _ = roc_curve(y2_test, y2_pred_proba)
axes[1].plot(fpr2, tpr2, label=f'XGBoost (AUC={roc_auc_score(y2_test, y2_pred_proba):.3f})')
axes[1].plot([0,1],[0,1],'k--', label='Aleatório')
axes[1].set_title('Curva ROC — Investigação', fontweight='bold')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend(loc='lower right')
plt.tight_layout(); plt.show()

## 5️⃣ Salvamento dos artefatos

**O que faz**: Persiste os dois modelos e suas listas de features para uso na API.

In [ ]:
os.makedirs('models', exist_ok=True)

joblib.dump(xgb_model,               'models/severity_classifier.pkl')
joblib.dump(xgb_model_2,             'models/investigation_classifier.pkl')
joblib.dump(X.columns.tolist(),      'models/features_severity.pkl')
joblib.dump(X2.columns.tolist(),     'models/features_investigation.pkl')

print('='*100)
print('💾 ARTEFATOS SALVOS')
print('='*100)
for f in ['models/severity_classifier.pkl','models/investigation_classifier.pkl',
          'models/features_severity.pkl','models/features_investigation.pkl']:
    size = os.path.getsize(f)/1024
    print(f'  ✓ {f:<45} ({size:.1f} KB)')

print('\n✅ PARTE 3 CONCLUÍDA — Modelos prontos para deploy!')

---
# 🚀 PARTE 4 — Deployment: API RESTful com Flask

**Fase CRISP-DM**: Deployment

Esta seção define, testa e documenta a API RESTful que expõe os dados e modelos para consumo por sistemas externos de gestão de risco.

**Endpoints disponíveis (9)**:

| Método | Rota | Descrição |
|---|---|---|
| GET | `/` | Documentação e status |
| GET | `/api/incidents` | Listar incidentes com filtros |
| GET | `/api/incidents/<id>` | Detalhe de um incidente |
| GET | `/api/stats/by-application` | Estatísticas por aplicação |
| GET | `/api/stats/by-segment` | Estatísticas por segmento |
| GET | `/api/stats/temporal` | Tendência temporal |
| GET | `/api/stats/governance` | Estatísticas de governança |
| POST | `/api/predict/severity` | Prediz severidade |
| POST | `/api/predict/investigation` | Prediz investigação |

---

## 1️⃣ Configuração e carregamento dos modelos

**O que faz**: Inicializa a aplicação Flask, conecta ao banco SQLite e carrega os modelos persistidos na Parte 3.

In [ ]:
DATABASE   = 'ai_finance_incidents.db'
MODELS_PATH = 'models'

# ── Helpers de banco ──────────────────────────────────────────────────────────
def get_db_connection():
    conn = sqlite3.connect(DATABASE)
    conn.row_factory = sqlite3.Row
    return conn

def query_db(query, args=(), one=False):
    conn = get_db_connection()
    cur  = conn.cursor()
    cur.execute(query, args)
    rows = cur.fetchall()
    conn.close()
    results = [dict(r) for r in rows]
    if one:
        return results[0] if results else None
    return results

# ── Carregar modelos ──────────────────────────────────────────────────────────
severity_model = investigation_model = None
features_severity = features_investigation = None
models_loaded = False

try:
    severity_model      = joblib.load(os.path.join(MODELS_PATH, 'severity_classifier.pkl'))
    investigation_model = joblib.load(os.path.join(MODELS_PATH, 'investigation_classifier.pkl'))
    features_severity   = joblib.load(os.path.join(MODELS_PATH, 'features_severity.pkl'))
    features_investigation = joblib.load(os.path.join(MODELS_PATH, 'features_investigation.pkl'))
    models_loaded = True
    print('✅ Modelos carregados.')
    print(f'   severity_classifier      → {type(severity_model).__name__}')
    print(f'   investigation_classifier → {type(investigation_model).__name__}')
    print(f'   Features (severity)      → {len(features_severity)}')
    print(f'   Features (investigation) → {len(features_investigation)}')
except Exception as e:
    print(f'⚠️ Modelos não carregados: {e}')
    print('   A API funcionará sem endpoints preditivos.')

## 2️⃣ Função auxiliar de pré-processamento

**O que faz**: Converte o payload JSON recebido pela API no vetor de features esperado pelo modelo.

In [ ]:
def prepare_model_input(data, expected_features, add_high_severity_default=False):
    """
    Transforma o payload da requisição no DataFrame de features do modelo.
    
    Parâmetros:
        data (dict)                   : payload JSON da requisição
        expected_features (list)      : lista de features do modelo (salva em .pkl)
        add_high_severity_default (bool): adiciona 'high_severity' se True
    
    Retorna:
        pd.DataFrame com shape (1, n_features)
    """
    normalized = {
        'application_type': data.get('application_type'),
        'incident_type':    data.get('incident_type'),
        'customer_segment': data.get('customer_segment'),
        'year':             data.get('year'),
        'fine_imposed':     data.get('fine_imposed', 0),
        'policy_change':    data.get('policy_change', 0),
        'third_party_audit':data.get('third_party_audit', 0),
    }

    df_in = pd.DataFrame([normalized])
    cat_cols = ['application_type','incident_type','customer_segment']
    df_enc = pd.get_dummies(df_in, columns=cat_cols, drop_first=True)

    if add_high_severity_default:
        df_enc['high_severity'] = data.get('high_severity', 0)

    # Garantir todas as features esperadas (preenche com 0 se ausente)
    for feat in expected_features:
        if feat not in df_enc.columns:
            df_enc[feat] = 0

    return df_enc[expected_features]

print('✅ Função prepare_model_input definida.')

## 3️⃣ Definição dos endpoints

**O que faz**: Define todas as rotas da API com lógica de consulta SQL e predição ML.

In [ ]:
app = Flask(__name__)
CORS(app)
app.config['JSON_SORT_KEYS']           = False
app.config['JSONIFY_PRETTYPRINT_REGULAR'] = True

# ── GET / ─────────────────────────────────────────────────────────────────────
@app.route('/', methods=['GET'])
def home():
    return jsonify({
        'api'         : 'AI Finance Incidents Analysis API',
        'version'     : '1.0.0',
        'methodology' : 'CRISP-DM — Deployment',
        'models_loaded': models_loaded,
        'timestamp'   : datetime.now().isoformat(),
        'endpoints'   : {
            'incidents'  : ['GET /api/incidents', 'GET /api/incidents/<id>'],
            'statistics' : ['GET /api/stats/by-application', 'GET /api/stats/by-segment',
                            'GET /api/stats/temporal', 'GET /api/stats/governance'],
            'predictions': ['POST /api/predict/severity', 'POST /api/predict/investigation']
        }
    })

# ── GET /api/incidents ────────────────────────────────────────────────────────
@app.route('/api/incidents', methods=['GET'])
def get_incidents():
    application_type = request.args.get('application_type')
    severity_level   = request.args.get('severity_level')
    year             = request.args.get('year')
    limit            = request.args.get('limit', 50, type=int)

    sql    = 'SELECT * FROM incidents WHERE 1=1'
    params = []
    if application_type:
        sql += ' AND application_type = ?'; params.append(application_type)
    if severity_level:
        sql += ' AND severity_level = ?';   params.append(severity_level)
    if year:
        sql += ' AND year = ?';             params.append(year)
    sql += f' LIMIT {limit}'

    incidents = query_db(sql, params)
    return jsonify({'total': len(incidents),
                    'filters': {'application_type': application_type,
                                'severity_level': severity_level,
                                'year': year, 'limit': limit},
                    'incidents': incidents})

# ── GET /api/incidents/<id> ───────────────────────────────────────────────────
@app.route('/api/incidents/<int:incident_id>', methods=['GET'])
def get_incident_detail(incident_id):
    incident = query_db('SELECT * FROM incidents WHERE incident_id = ?',
                        (incident_id,), one=True)
    if not incident:
        return jsonify({'error': 'Incidente não encontrado'}), 404
    fi = query_db('SELECT * FROM financial_impacts WHERE incident_id = ?',
                  (incident_id,), one=True)
    rr = query_db('SELECT * FROM regulatory_responses WHERE incident_id = ?',
                  (incident_id,), one=True)
    return jsonify({'incident': incident, 'financial_impact': fi, 'regulatory_response': rr})

# ── GET /api/stats/by-application ────────────────────────────────────────────
@app.route('/api/stats/by-application', methods=['GET'])
def stats_by_application():
    q = '''
    SELECT application_type,
           COUNT(*) AS total_incidents,
           SUM(CASE WHEN severity_level IN ("high","critical") THEN 1 ELSE 0 END) AS high_severity_count,
           ROUND(AVG(CASE WHEN severity_level="critical" THEN 4
                          WHEN severity_level="high" THEN 3
                          WHEN severity_level="medium" THEN 2 ELSE 1 END), 2) AS avg_severity_score
    FROM incidents
    GROUP BY application_type ORDER BY total_incidents DESC'''
    results = query_db(q)
    return jsonify({'statistics_by_application': results, 'total_categories': len(results)})

# ── GET /api/stats/by-segment ─────────────────────────────────────────────────
@app.route('/api/stats/by-segment', methods=['GET'])
def stats_by_segment():
    q = '''
    SELECT customer_segment,
           COUNT(*) AS total_incidents,
           SUM(CASE WHEN incident_type="algorithmic_bias" THEN 1 ELSE 0 END) AS bias_incidents,
           ROUND(100.0 * SUM(CASE WHEN incident_type="algorithmic_bias" THEN 1 ELSE 0 END)
                        / COUNT(*), 2) AS bias_percentage
    FROM incidents GROUP BY customer_segment ORDER BY total_incidents DESC'''
    return jsonify({'statistics_by_segment': query_db(q)})

# ── GET /api/stats/temporal ───────────────────────────────────────────────────
@app.route('/api/stats/temporal', methods=['GET'])
def stats_temporal():
    q = '''
    SELECT year, COUNT(*) AS total_incidents,
           SUM(CASE WHEN severity_level IN ("high","critical") THEN 1 ELSE 0 END) AS high_severity,
           SUM(CASE WHEN incident_type="algorithmic_bias" THEN 1 ELSE 0 END) AS bias_incidents
    FROM incidents WHERE year IS NOT NULL GROUP BY year ORDER BY year'''
    results = query_db(q)
    period  = f"{results[0]['year']} – {results[-1]['year']}" if results else None
    return jsonify({'temporal_statistics': results, 'period': period})

# ── GET /api/stats/governance ─────────────────────────────────────────────────
@app.route('/api/stats/governance', methods=['GET'])
def stats_governance():
    q = '''
    SELECT SUM(regulatory_investigation) AS total_investigations,
           SUM(fine_imposed) AS total_fines,
           SUM(policy_change) AS total_policy_changes,
           SUM(third_party_audit) AS total_audits,
           COUNT(*) AS total_incidents
    FROM regulatory_responses'''
    res   = query_db(q, one=True)
    total = res.get('total_incidents', 0) or 1
    rates = {k: round(100.0*(res.get(v) or 0)/total, 2)
             for k, v in [('investigation_rate','total_investigations'),
                           ('fine_rate','total_fines'),
                           ('policy_change_rate','total_policy_changes'),
                           ('audit_rate','total_audits')]}
    return jsonify({'governance_statistics': res, 'rates_percentage': rates})

# ── POST /api/predict/severity ────────────────────────────────────────────────
@app.route('/api/predict/severity', methods=['POST'])
def predict_severity():
    if not models_loaded:
        return jsonify({'error': 'Modelos ML não carregados no servidor.'}), 503
    try:
        data     = request.get_json()
        X_in     = prepare_model_input(data, features_severity, add_high_severity_default=False)
        pred     = severity_model.predict(X_in)[0]
        proba    = severity_model.predict_proba(X_in)[0]
        severity = 'high' if pred == 1 else 'low'
        prob_val = float(proba[pred])
        conf     = 'high' if prob_val >= 0.75 else 'medium' if prob_val >= 0.55 else 'low'
        return jsonify({'prediction': severity, 'probability': round(prob_val, 4),
                        'confidence': conf,
                        'interpretation': f'Severidade: {severity.upper()} | confiança: {conf}'})
    except Exception as e:
        return jsonify({'error': str(e)}), 400

# ── POST /api/predict/investigation ──────────────────────────────────────────
@app.route('/api/predict/investigation', methods=['POST'])
def predict_investigation():
    if not models_loaded:
        return jsonify({'error': 'Modelos ML não carregados no servidor.'}), 503
    try:
        data         = request.get_json()
        X_in         = prepare_model_input(data, features_investigation, add_high_severity_default=True)
        pred         = investigation_model.predict(X_in)[0]
        proba        = investigation_model.predict_proba(X_in)[0]
        will_invest  = bool(pred)
        prob_val     = float(proba[1])
        risk         = 'high' if prob_val >= 0.70 else 'medium' if prob_val >= 0.40 else 'low'
        rec          = ('Preparar documentação, trilha de auditoria e evidências de compliance.'
                        if will_invest else
                        'Manter monitoramento padrão e acompanhamento preventivo.')
        return jsonify({'will_be_investigated': will_invest, 'probability': round(prob_val, 4),
                        'risk_level': risk, 'recommendation': rec})
    except Exception as e:
        return jsonify({'error': str(e)}), 400

print('✅ Todos os 9 endpoints definidos.')

## 4️⃣ Testes automatizados

**O que faz**: Executa testes de sanidade em todos os endpoints usando o cliente de teste do Flask.

In [ ]:
app.config['TESTING'] = True
client = app.test_client()

print('='*100)
print('🧪 TESTES AUTOMATIZADOS DA API')
print('='*100)

SAMPLE_SEVERITY = {
    'application_type': 'credit_scoring', 'incident_type': 'algorithmic_bias',
    'customer_segment': 'retail', 'year': 2024,
    'fine_imposed': 0, 'policy_change': 0, 'third_party_audit': 0
}
SAMPLE_INVESTIGATION = {**SAMPLE_SEVERITY, 'high_severity': 1}

tests = [
    ('GET',  '/',                          None,                  'Documentação/status'),
    ('GET',  '/api/incidents?limit=5',     None,                  'Listar incidentes'),
    ('GET',  '/api/stats/by-application',  None,                  'Stats por aplicação'),
    ('GET',  '/api/stats/by-segment',      None,                  'Stats por segmento'),
    ('GET',  '/api/stats/temporal',        None,                  'Stats temporal'),
    ('GET',  '/api/stats/governance',      None,                  'Stats governança'),
    ('POST', '/api/predict/severity',      SAMPLE_SEVERITY,       'Predizer severidade'),
    ('POST', '/api/predict/investigation', SAMPLE_INVESTIGATION,  'Predizer investigação'),
]

all_ok = True
for method, route, payload, desc in tests:
    if method == 'GET':
        resp = client.get(route)
    else:
        resp = client.post(route, json=payload)
    ok = resp.status_code in (200, 503)  # 503 aceitável se modelos não carregados
    icon = '✅' if ok else '❌'
    if not ok:
        all_ok = False
    body = resp.get_json()
    preview = str(body)[:80] if body else '-'
    print(f'{icon} [{resp.status_code}] {method:4s} {route:<45} → {preview}')

print('\n' + ('✅ TODOS OS TESTES PASSARAM!' if all_ok else '⚠️ ALGUNS TESTES FALHARAM.'))

## 5️⃣ Arquivo app.py standalone

**O que faz**: Gera o arquivo `app.py` pronto para deploy em servidor de produção.

In [ ]:
APP_PY_CONTENT = '''# -*- coding: utf-8 -*-
"""
AI Finance Incidents Analysis API — app.py
API RESTful para análise de incidentes de IA em serviços financeiros.
Projeto Integrador — PUC-SP | Metodologia: CRISP-DM

Execução local:
    python app.py

Produção (Gunicorn):
    gunicorn -w 4 -b 0.0.0.0:8000 app:app
"""

import os, sqlite3, joblib
import pandas as pd
from flask import Flask, request, jsonify
from flask_cors import CORS
from datetime import datetime

DATABASE    = "ai_finance_incidents.db"
MODELS_PATH = "models"

def get_db_connection():
    conn = sqlite3.connect(DATABASE)
    conn.row_factory = sqlite3.Row
    return conn

def query_db(query, args=(), one=False):
    conn = get_db_connection()
    cur  = conn.cursor()
    cur.execute(query, args)
    rows = cur.fetchall()
    conn.close()
    results = [dict(r) for r in rows]
    return results[0] if (one and results) else results

severity_model = investigation_model = None
features_severity = features_investigation = None
models_loaded = False
try:
    severity_model      = joblib.load(os.path.join(MODELS_PATH, "severity_classifier.pkl"))
    investigation_model = joblib.load(os.path.join(MODELS_PATH, "investigation_classifier.pkl"))
    features_severity   = joblib.load(os.path.join(MODELS_PATH, "features_severity.pkl"))
    features_investigation = joblib.load(os.path.join(MODELS_PATH, "features_investigation.pkl"))
    models_loaded = True
except Exception:
    models_loaded = False

app = Flask(__name__)
CORS(app)
app.config["JSON_SORT_KEYS"] = False
app.config["JSONIFY_PRETTYPRINT_REGULAR"] = True

def prepare_model_input(data, expected_features, add_high_severity_default=False):
    normalized = {
        "application_type": data.get("application_type"),
        "incident_type"   : data.get("incident_type"),
        "customer_segment": data.get("customer_segment"),
        "year"            : data.get("year"),
        "fine_imposed"    : data.get("fine_imposed", 0),
        "policy_change"   : data.get("policy_change", 0),
        "third_party_audit": data.get("third_party_audit", 0),
    }
    df_in = pd.DataFrame([normalized])
    df_enc = pd.get_dummies(df_in, columns=["application_type","incident_type","customer_segment"], drop_first=True)
    if add_high_severity_default:
        df_enc["high_severity"] = data.get("high_severity", 0)
    for feat in expected_features:
        if feat not in df_enc.columns:
            df_enc[feat] = 0
    return df_enc[expected_features]

@app.route("/", methods=["GET"])
def home():
    return jsonify({"api":"AI Finance Incidents API","version":"1.0.0",
                    "models_loaded":models_loaded,"timestamp":datetime.now().isoformat()})

@app.route("/api/incidents", methods=["GET"])
def get_incidents():
    application_type = request.args.get("application_type")
    severity_level   = request.args.get("severity_level")
    year             = request.args.get("year")
    limit            = request.args.get("limit", 50, type=int)
    sql, params = "SELECT * FROM incidents WHERE 1=1", []
    if application_type: sql += " AND application_type = ?"; params.append(application_type)
    if severity_level:   sql += " AND severity_level = ?";   params.append(severity_level)
    if year:             sql += " AND year = ?";             params.append(year)
    sql += f" LIMIT {limit}"
    return jsonify({"total": len(query_db(sql, params)), "incidents": query_db(sql, params)})

@app.route("/api/incidents/<int:incident_id>", methods=["GET"])
def get_incident_detail(incident_id):
    inc = query_db("SELECT * FROM incidents WHERE incident_id=?", (incident_id,), one=True)
    if not inc: return jsonify({"error":"Incidente não encontrado"}), 404
    return jsonify({"incident":inc,
                    "financial_impact": query_db("SELECT * FROM financial_impacts WHERE incident_id=?",(incident_id,),one=True),
                    "regulatory_response": query_db("SELECT * FROM regulatory_responses WHERE incident_id=?",(incident_id,),one=True)})

@app.route("/api/stats/by-application", methods=["GET"])
def stats_by_application():
    return jsonify({"statistics_by_application": query_db(
        "SELECT application_type, COUNT(*) AS total FROM incidents GROUP BY application_type ORDER BY total DESC")})

@app.route("/api/stats/by-segment", methods=["GET"])
def stats_by_segment():
    return jsonify({"statistics_by_segment": query_db(
        "SELECT customer_segment, COUNT(*) AS total FROM incidents GROUP BY customer_segment ORDER BY total DESC")})

@app.route("/api/stats/temporal", methods=["GET"])
def stats_temporal():
    return jsonify({"temporal_statistics": query_db(
        "SELECT year, COUNT(*) AS total FROM incidents WHERE year IS NOT NULL GROUP BY year ORDER BY year")})

@app.route("/api/stats/governance", methods=["GET"])
def stats_governance():
    return jsonify({"governance": query_db(
        "SELECT SUM(regulatory_investigation) AS investigations, SUM(fine_imposed) AS fines FROM regulatory_responses", one=True)})

@app.route("/api/predict/severity", methods=["POST"])
def predict_severity():
    if not models_loaded: return jsonify({"error":"Modelos não carregados"}), 503
    try:
        data = request.get_json()
        X_in = prepare_model_input(data, features_severity, False)
        pred = severity_model.predict(X_in)[0]
        prob = float(severity_model.predict_proba(X_in)[0][pred])
        return jsonify({"prediction":"high" if pred else "low","probability":round(prob,4)})
    except Exception as e: return jsonify({"error":str(e)}), 400

@app.route("/api/predict/investigation", methods=["POST"])
def predict_investigation():
    if not models_loaded: return jsonify({"error":"Modelos não carregados"}), 503
    try:
        data = request.get_json()
        X_in = prepare_model_input(data, features_investigation, True)
        pred = investigation_model.predict(X_in)[0]
        prob = float(investigation_model.predict_proba(X_in)[0][1])
        return jsonify({"will_be_investigated":bool(pred),"probability":round(prob,4)})
    except Exception as e: return jsonify({"error":str(e)}), 400

if __name__ == "__main__":
    print("API iniciando em http://localhost:5000")
    app.run(host="0.0.0.0", port=5000, debug=True)
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(APP_PY_CONTENT)

print('='*100)
print('💾 app.py GERADO COM SUCESSO')
print('='*100)
print('\nPara executar:')
print('  python app.py')
print('\nPara produção:')
print('  gunicorn -w 4 -b 0.0.0.0:8000 app:app')
print('\n✅ PARTE 4 CONCLUÍDA — API RESTful completa!')

---
# 🎉 Projeto Completo — Resumo Final

## ✅ CRISP-DM Executado com Sucesso

| Fase | Status | Entregável |
|---|---|---|
| Data Understanding | ✅ | Exploração inicial + qualidade |
| Data Preparation | ✅ | `incidents_finance_filtered.csv` + `ai_finance_incidents.db` |
| Evaluation | ✅ | 4 testes de hipóteses estatísticos |
| Modeling | ✅ | `models/severity_classifier.pkl` + `models/investigation_classifier.pkl` |
| Deployment | ✅ | API RESTful com 9 endpoints + `app.py` |

## 📦 Artefatos gerados

```
incidents_finance_filtered.csv    ← dataset processado
ai_finance_incidents.db           ← banco SQLite (3 tabelas)
models/severity_classifier.pkl    ← modelo de severidade
models/investigation_classifier.pkl  ← modelo de investigação
models/features_severity.pkl      ← lista de features (M1)
models/features_investigation.pkl ← lista de features (M2)
app.py                            ← API RESTful standalone
```

---


In [ ]:
print('='*80)
print('🎉 NOTEBOOK UNIFICADO — CONCLUÍDO COM SUCESSO!')
print('='*80)
print(f'📊 Dataset processado : {len(df_final):,} incidentes')
print(f'🗄️  Banco de dados     : 3 tabelas relacionadas')
print(f'🤖 Modelos ML         : 2 classificadores XGBoost')
print(f'🚀 API RESTful        : 9 endpoints')
print()
print('Fonte de dados: AI Incident Database (AIID) — https://incidentdatabase.ai')
print('Metodologia   : CRISP-DM — PUC-SP')